# Paired Significance Test — Pemilihan Config (Kualitas Linguistik)

Membandingkan tiga config pada setting **2000 data / 10 epoch** (method3): `margin_part`,
`margin_full`, `finetune`. Karena ketiganya dievaluasi pada **dialog yang sama**, kita pakai
**paired test** (Wilcoxon signed-rank + paired t) pada skor UniEval **per-instance** — jauh lebih
kuat daripada membandingkan rata-rata, sebab mengontrol kesulitan tiap dialog.

Fokus: metrik kualitas linguistik (coherence, relevance, fluency, consistency) + skor komposit.

> **Sebelum menjalankan, verifikasi dua hal:**
> 1. **Label config benar** — pastikan `method3` memang 2000d/10ep (di notebook hasil ada
>    inkonsistensi suffix `_400`/`_600` vs komentar sel). Sesuaikan `METHOD` & path bila perlu.
> 2. **File `unieval.json` berisi list per-instance** (satu dict skor per dialog), bukan rata-rata.

## 1. Loader minimal (mandiri)

In [9]:
import json
from pathlib import Path

def load_json(path):
    p = Path(path)
    if not p.is_file():
        raise FileNotFoundError(f"Tidak ditemukan: {p}")
    with open(p, "r", encoding="utf-8") as f:
        return json.load(f)

## 2. Paired Test

In [10]:
# ============================================================
#  PAIRED SIGNIFICANCE TEST — pilih config terbaik (kualitas linguistik)
#  Membandingkan config 2000d/10ep: margin_part vs margin_full vs finetune
#  Memakai skor UniEval PER-INSTANCE (per dialog yang sama) -> paired test.
#  Butuh: load_json() yang sudah ada di notebook.
# ============================================================
import numpy as np, pandas as pd
from scipy.stats import wilcoxon, ttest_rel
from itertools import combinations

# --- config yang dibandingkan: (method_num, folder) pada setting 2000d/10ep (method3) ---
METHOD = "method1"
CONFIGS = {
    "margin_part": f"{METHOD}_outputs/bart_margin_part",
    "margin_full": f"{METHOD}_outputs/bart_margin_full",
    "finetune":    f"{METHOD}_outputs/bart_finetune",
}
LING = ["rouge1","bertscore_f1","coherence", "relevance", "fluency", "consistency", "alignscore"]

def load_per_instance(folder):
    """Muat unieval.json (list per-instance) -> DataFrame per dialog."""
    data = load_json(f"{folder}/unieval.json")
    align_score = load_json(f"{folder}/alignscore.json")
    rouge = load_json(f"{folder}/generated.json")

    rouge_data = [{"rouge1" : r.get("rouge1"), "bertscore_f1" : r.get("bertscore_f1")} for r in rouge]

    # Pastikan ketiga list memiliki panjang yang sama
    assert len(data) == len(align_score) == len(rouge), (
        f"Length mismatch: data={len(data)}, align_score={len(align_score)}, rouge={len(rouge)}"
    )

    rows = [
        {
            **row, 
            "alignscore": score,
            "rouge1": r.get("rouge1"),
            "bertscore_f1": r.get("bertscore_f1")
        }
        for row, score, r in zip(data, align_score, rouge)
    ]

    return pd.DataFrame(rows)

# muat semua config
per = {name: load_per_instance(folder) for name, folder in CONFIGS.items()}
n = min(len(d) for d in per.values())
print("Jumlah dialog per config:", {k: len(v) for k, v in per.items()}, "-> pakai", n, "(paired)")
for k in per: per[k] = per[k].iloc[:n].reset_index(drop=True)

# --- skor komposit linguistik (rata-rata 4 metrik) per dialog ---
for k in per:
    per[k]["_ling"] = per[k][LING].mean(axis=1)

# --- ringkasan mean ---
print("\n" + "="*64)
print("MEAN per config (2000d/10ep)")
print("="*64)
summ = pd.DataFrame({k: per[k][LING + ["_ling"]].mean() for k in per}).T
print(summ.round(4).to_string())

# --- paired test tiap pasangan config, tiap metrik + komposit ---
def effect_size(d):  # Cohen's dz utk paired
    return d.mean() / d.std(ddof=1) if d.std(ddof=1) > 0 else 0.0

print("\n" + "="*64)
print("PAIRED TEST (Wilcoxon signed-rank; + paired t sebagai pembanding)")
print("="*64)
for a, b in combinations(per, 2):
    print(f"\n>>> {a}  vs  {b}")
    for m in LING + ["_ling"]:
        x, y = per[a][m].values, per[b][m].values
        d = x - y
        if np.allclose(d, 0):
            print(f"  {m:12s} identik"); continue
        try:
            w, pw = wilcoxon(x, y)
        except ValueError:
            pw = np.nan
        t, pt = ttest_rel(x, y)
        winner = a if d.mean() > 0 else b
        sig = "**SIG**" if (pw < 0.05) else "ns"
        print(f"  {m:12s} Δmean={d.mean():+.4f}  dz={effect_size(d):+.3f}  "
              f"Wilcoxon p={pw:.3g} [{sig}]  (unggul: {winner})")

print("""
CARA BACA:
- Δmean>0 berarti config pertama lebih tinggi. 'SIG' = beda signifikan (p<0.05) pada dialog berpasangan.
- dz = effect size (Cohen's dz): |0.2| kecil, |0.5| sedang, |0.8| besar. p kecil tapi dz kecil = beda nyata tapi sepele.
- Untuk klaim 'config X terbaik secara kualitas linguistik', idealnya X unggul SIGNIFIKAN di mayoritas
  metrik LING dan pada skor komposit _ling, vs SEMUA pembanding.
- Jika margin_part vs margin_full tidak signifikan di _ling -> laporkan keduanya setara, pilih atas dasar lain
  (mis. align/faktualitas) dan sebutkan di paper bahwa selisihnya tidak signifikan.""")

# --- ekspor ringkas ---
summ.round(4).to_csv("config_comparison_2000d10ep.csv")
print("\n>> Disimpan: config_comparison_2000d10ep.csv")


Jumlah dialog per config: {'margin_part': 500, 'margin_full': 500, 'finetune': 500} -> pakai 500 (paired)

MEAN per config (2000d/10ep)
             rouge1  bertscore_f1  coherence  relevance  fluency  consistency  alignscore   _ling
margin_part  0.5662        0.9049     0.8257     0.7557   0.7963       0.8463      0.7053  0.7715
margin_full  0.5741        0.9054     0.8130     0.7495   0.7862       0.8431      0.6855  0.7653
finetune     0.5942        0.9087     0.8205     0.7680   0.7731       0.8519      0.7087  0.7750

PAIRED TEST (Wilcoxon signed-rank; + paired t sebagai pembanding)

>>> margin_part  vs  margin_full
  rouge1       Δmean=-0.0079  dz=-0.123  Wilcoxon p=0.0207 [**SIG**]  (unggul: margin_full)
  bertscore_f1 Δmean=-0.0006  dz=-0.054  Wilcoxon p=0.5 [ns]  (unggul: margin_full)
  coherence    Δmean=+0.0127  dz=+0.046  Wilcoxon p=0.437 [ns]  (unggul: margin_part)
  relevance    Δmean=+0.0062  dz=+0.020  Wilcoxon p=0.616 [ns]  (unggul: margin_part)
  fluency      Δmean=+0

In [11]:
METHOD = "method2"
CONFIGS = {
    "margin_part": f"{METHOD}_outputs/bart_margin_part",
    "margin_full": f"{METHOD}_outputs/bart_margin_full",
    "finetune":    f"{METHOD}_outputs/bart_finetune",
}

# muat semua config
per = {name: load_per_instance(folder) for name, folder in CONFIGS.items()}
n = min(len(d) for d in per.values())
print("Jumlah dialog per config:", {k: len(v) for k, v in per.items()}, "-> pakai", n, "(paired)")
for k in per: per[k] = per[k].iloc[:n].reset_index(drop=True)

# --- skor komposit linguistik (rata-rata 4 metrik) per dialog ---
for k in per:
    per[k]["_ling"] = per[k][LING].mean(axis=1)

# --- ringkasan mean ---
print("\n" + "="*64)
print("MEAN per config (2000d/10ep)")
print("="*64)
summ = pd.DataFrame({k: per[k][LING + ["_ling"]].mean() for k in per}).T
print(summ.round(4).to_string())

# --- paired test tiap pasangan config, tiap metrik + komposit ---
def effect_size(d):  # Cohen's dz utk paired
    return d.mean() / d.std(ddof=1) if d.std(ddof=1) > 0 else 0.0

print("\n" + "="*64)
print("PAIRED TEST (Wilcoxon signed-rank; + paired t sebagai pembanding)")
print("="*64)
for a, b in combinations(per, 2):
    print(f"\n>>> {a}  vs  {b}")
    for m in LING + ["_ling"]:
        x, y = per[a][m].values, per[b][m].values
        d = x - y
        if np.allclose(d, 0):
            print(f"  {m:12s} identik"); continue
        try:
            w, pw = wilcoxon(x, y)
        except ValueError:
            pw = np.nan
        t, pt = ttest_rel(x, y)
        winner = a if d.mean() > 0 else b
        sig = "**SIG**" if (pw < 0.05) else "ns"
        print(f"  {m:12s} Δmean={d.mean():+.4f}  dz={effect_size(d):+.3f}  "
              f"Wilcoxon p={pw:.3g} [{sig}]  (unggul: {winner})")

Jumlah dialog per config: {'margin_part': 500, 'margin_full': 500, 'finetune': 500} -> pakai 500 (paired)

MEAN per config (2000d/10ep)
             rouge1  bertscore_f1  coherence  relevance  fluency  consistency  alignscore   _ling
margin_part  0.6039        0.9084     0.8565     0.8259   0.8132       0.8672      0.7470  0.8031
margin_full  0.6098        0.9101     0.8576     0.8297   0.8025       0.8733      0.7395  0.8032
finetune     0.6140        0.9107     0.8523     0.8232   0.8042       0.8652      0.7454  0.8022

PAIRED TEST (Wilcoxon signed-rank; + paired t sebagai pembanding)

>>> margin_part  vs  margin_full
  rouge1       Δmean=-0.0059  dz=-0.090  Wilcoxon p=0.0397 [**SIG**]  (unggul: margin_full)
  bertscore_f1 Δmean=-0.0017  dz=-0.133  Wilcoxon p=0.00252 [**SIG**]  (unggul: margin_full)
  coherence    Δmean=-0.0011  dz=-0.005  Wilcoxon p=0.939 [ns]  (unggul: margin_full)
  relevance    Δmean=-0.0038  dz=-0.015  Wilcoxon p=0.592 [ns]  (unggul: margin_full)
  fluency     

In [12]:
METHOD = "method3"
CONFIGS = {
    "margin_part": f"{METHOD}_outputs/bart_margin_part",
    "margin_full": f"{METHOD}_outputs/bart_margin_full",
    "finetune":    f"{METHOD}_outputs/bart_finetune",
}

# muat semua config
per = {name: load_per_instance(folder) for name, folder in CONFIGS.items()}
n = min(len(d) for d in per.values())
print("Jumlah dialog per config:", {k: len(v) for k, v in per.items()}, "-> pakai", n, "(paired)")
for k in per: per[k] = per[k].iloc[:n].reset_index(drop=True)

# --- skor komposit linguistik (rata-rata 4 metrik) per dialog ---
for k in per:
    per[k]["_ling"] = per[k][LING].mean(axis=1)

# --- ringkasan mean ---
print("\n" + "="*64)
print("MEAN per config (2000d/10ep)")
print("="*64)
summ = pd.DataFrame({k: per[k][LING + ["_ling"]].mean() for k in per}).T
print(summ.round(4).to_string())

# --- paired test tiap pasangan config, tiap metrik + komposit ---
def effect_size(d):  # Cohen's dz utk paired
    return d.mean() / d.std(ddof=1) if d.std(ddof=1) > 0 else 0.0

print("\n" + "="*64)
print("PAIRED TEST (Wilcoxon signed-rank; + paired t sebagai pembanding)")
print("="*64)
for a, b in combinations(per, 2):
    print(f"\n>>> {a}  vs  {b}")
    for m in LING + ["_ling"]:
        x, y = per[a][m].values, per[b][m].values
        d = x - y
        if np.allclose(d, 0):
            print(f"  {m:12s} identik"); continue
        try:
            w, pw = wilcoxon(x, y)
        except ValueError:
            pw = np.nan
        t, pt = ttest_rel(x, y)
        winner = a if d.mean() > 0 else b
        sig = "**SIG**" if (pw < 0.05) else "ns"
        print(f"  {m:12s} Δmean={d.mean():+.4f}  dz={effect_size(d):+.3f}  "
              f"Wilcoxon p={pw:.3g} [{sig}]  (unggul: {winner})")

Jumlah dialog per config: {'margin_part': 500, 'margin_full': 500, 'finetune': 500} -> pakai 500 (paired)

MEAN per config (2000d/10ep)
             rouge1  bertscore_f1  coherence  relevance  fluency  consistency  alignscore   _ling
margin_part  0.6156        0.9108     0.8885     0.8587   0.8334       0.8796      0.7774  0.8234
margin_full  0.6199        0.9111     0.8952     0.8608   0.8267       0.8790      0.7587  0.8216
finetune     0.6214        0.9124     0.8796     0.8420   0.8187       0.8756      0.7603  0.8157

PAIRED TEST (Wilcoxon signed-rank; + paired t sebagai pembanding)

>>> margin_part  vs  margin_full
  rouge1       Δmean=-0.0043  dz=-0.068  Wilcoxon p=0.235 [ns]  (unggul: margin_full)
  bertscore_f1 Δmean=-0.0003  dz=-0.027  Wilcoxon p=0.974 [ns]  (unggul: margin_full)
  coherence    Δmean=-0.0067  dz=-0.031  Wilcoxon p=0.382 [ns]  (unggul: margin_full)
  relevance    Δmean=-0.0021  dz=-0.008  Wilcoxon p=0.918 [ns]  (unggul: margin_full)
  fluency      Δmean=+0.006

In [13]:
METHOD = "method4"
CONFIGS = {
    "margin_part": f"{METHOD}_outputs/bart_margin_part",
    "margin_full": f"{METHOD}_outputs/bart_margin_full",
    "finetune":    f"{METHOD}_outputs/bart_finetune",
}

# muat semua config
per = {name: load_per_instance(folder) for name, folder in CONFIGS.items()}
n = min(len(d) for d in per.values())
print("Jumlah dialog per config:", {k: len(v) for k, v in per.items()}, "-> pakai", n, "(paired)")
for k in per: per[k] = per[k].iloc[:n].reset_index(drop=True)

# --- skor komposit linguistik (rata-rata 4 metrik) per dialog ---
for k in per:
    per[k]["_ling"] = per[k][LING].mean(axis=1)

# --- ringkasan mean ---
print("\n" + "="*64)
print("MEAN per config (2000d/10ep)")
print("="*64)
summ = pd.DataFrame({k: per[k][LING + ["_ling"]].mean() for k in per}).T
print(summ.round(4).to_string())

# --- paired test tiap pasangan config, tiap metrik + komposit ---
def effect_size(d):  # Cohen's dz utk paired
    return d.mean() / d.std(ddof=1) if d.std(ddof=1) > 0 else 0.0

print("\n" + "="*64)
print("PAIRED TEST (Wilcoxon signed-rank; + paired t sebagai pembanding)")
print("="*64)
for a, b in combinations(per, 2):
    print(f"\n>>> {a}  vs  {b}")
    for m in LING + ["_ling"]:
        x, y = per[a][m].values, per[b][m].values
        d = x - y
        if np.allclose(d, 0):
            print(f"  {m:12s} identik"); continue
        try:
            w, pw = wilcoxon(x, y)
        except ValueError:
            pw = np.nan
        t, pt = ttest_rel(x, y)
        winner = a if d.mean() > 0 else b
        sig = "**SIG**" if (pw < 0.05) else "ns"
        print(f"  {m:12s} Δmean={d.mean():+.4f}  dz={effect_size(d):+.3f}  "
              f"Wilcoxon p={pw:.3g} [{sig}]  (unggul: {winner})")

Jumlah dialog per config: {'margin_part': 500, 'margin_full': 500, 'finetune': 500} -> pakai 500 (paired)

MEAN per config (2000d/10ep)
             rouge1  bertscore_f1  coherence  relevance  fluency  consistency  alignscore   _ling
margin_part  0.6153        0.9110     0.8961     0.8839   0.8357       0.8920      0.7999  0.8334
margin_full  0.6259        0.9129     0.8823     0.8646   0.8173       0.8827      0.7846  0.8243
finetune     0.6238        0.9130     0.8604     0.8386   0.8057       0.8791      0.7865  0.8153

PAIRED TEST (Wilcoxon signed-rank; + paired t sebagai pembanding)

>>> margin_part  vs  margin_full
  rouge1       Δmean=-0.0106  dz=-0.168  Wilcoxon p=0.00371 [**SIG**]  (unggul: margin_full)
  bertscore_f1 Δmean=-0.0019  dz=-0.156  Wilcoxon p=0.0266 [**SIG**]  (unggul: margin_full)
  coherence    Δmean=+0.0138  dz=+0.063  Wilcoxon p=0.268 [ns]  (unggul: margin_part)
  relevance    Δmean=+0.0193  dz=+0.084  Wilcoxon p=0.0682 [ns]  (unggul: margin_part)
  fluency    

In [14]:
METHOD = "method6"
CONFIGS = {
    "margin_part": f"{METHOD}_outputs/qwen_5_margin_part",
    "margin_full": f"{METHOD}_outputs/qwen_5_margin_full",
    "finetune":    f"{METHOD}_outputs/qwen_5_finetune",
}

# muat semua config
per = {name: load_per_instance(folder) for name, folder in CONFIGS.items()}
n = min(len(d) for d in per.values())
print("Jumlah dialog per config:", {k: len(v) for k, v in per.items()}, "-> pakai", n, "(paired)")
for k in per: per[k] = per[k].iloc[:n].reset_index(drop=True)

# --- skor komposit linguistik (rata-rata 4 metrik) per dialog ---
for k in per:
    per[k]["_ling"] = per[k][LING].mean(axis=1)

# --- ringkasan mean ---
print("\n" + "="*64)
print("MEAN per config (2000d/10ep)")
print("="*64)
summ = pd.DataFrame({k: per[k][LING + ["_ling"]].mean() for k in per}).T
print(summ.round(4).to_string())

# --- paired test tiap pasangan config, tiap metrik + komposit ---
def effect_size(d):  # Cohen's dz utk paired
    return d.mean() / d.std(ddof=1) if d.std(ddof=1) > 0 else 0.0

print("\n" + "="*64)
print("PAIRED TEST (Wilcoxon signed-rank; + paired t sebagai pembanding)")
print("="*64)
for a, b in combinations(per, 2):
    print(f"\n>>> {a}  vs  {b}")
    for m in LING + ["_ling"]:
        x, y = per[a][m].values, per[b][m].values
        d = x - y
        if np.allclose(d, 0):
            print(f"  {m:12s} identik"); continue
        try:
            w, pw = wilcoxon(x, y)
        except ValueError:
            pw = np.nan
        t, pt = ttest_rel(x, y)
        winner = a if d.mean() > 0 else b
        sig = "**SIG**" if (pw < 0.05) else "ns"
        print(f"  {m:12s} Δmean={d.mean():+.4f}  dz={effect_size(d):+.3f}  "
              f"Wilcoxon p={pw:.3g} [{sig}]  (unggul: {winner})")

Jumlah dialog per config: {'margin_part': 500, 'margin_full': 500, 'finetune': 500} -> pakai 500 (paired)

MEAN per config (2000d/10ep)
             rouge1  bertscore_f1  coherence  relevance  fluency  consistency  alignscore   _ling
margin_part  0.5069        0.8907     0.8840     0.8635   0.8638       0.8163      0.7233  0.7926
margin_full  0.4339        0.8610     0.7325     0.6586   0.6424       0.7926      0.7408  0.6946
finetune     0.5041        0.8895     0.8838     0.8631   0.8572       0.8145      0.7307  0.7919

PAIRED TEST (Wilcoxon signed-rank; + paired t sebagai pembanding)

>>> margin_part  vs  margin_full
  rouge1       Δmean=+0.0729  dz=+0.417  Wilcoxon p=5.02e-18 [**SIG**]  (unggul: margin_part)
  bertscore_f1 Δmean=+0.0297  dz=+0.690  Wilcoxon p=2.84e-41 [**SIG**]  (unggul: margin_part)
  coherence    Δmean=+0.1515  dz=+0.524  Wilcoxon p=4.63e-30 [**SIG**]  (unggul: margin_part)
  relevance    Δmean=+0.2049  dz=+0.628  Wilcoxon p=5.03e-35 [**SIG**]  (unggul: margin_p